In [79]:
import pandas as pd

In [80]:
train = pd.read_csv('twitter_training.csv')

In [81]:
train

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...
...,...,...,...,...
74676,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74677,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74678,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74679,9200,Nvidia,Positive,Just realized between the windows partition of...


In [82]:
## Can work with 15000 samples for faster training

# train.sample(15000, random_state=42)

In [83]:
train.shape

(74681, 4)

In [84]:
train.columns

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')

In [85]:
train['Positive'].value_counts()

Positive
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64

In [86]:
train.isnull().sum()

2401                                                       0
Borderlands                                                0
Positive                                                   0
im getting on borderlands and i will murder you all ,    686
dtype: int64

In [87]:
train.duplicated().sum()

2700

### Data Cleaning 

In [88]:
train = train.dropna()

In [89]:
train.isnull().sum()

2401                                                     0
Borderlands                                              0
Positive                                                 0
im getting on borderlands and i will murder you all ,    0
dtype: int64

In [90]:
train.shape

(73995, 4)

In [91]:
train.drop_duplicates(inplace=True)

/tmp/ipykernel_88243/1644190799.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train.drop_duplicates(inplace=True)


In [92]:
train.duplicated().sum()

0

In [93]:
train.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [94]:
train.columns

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')

In [95]:
train.drop(columns=['2401','Borderlands'], inplace=True)

/tmp/ipykernel_88243/1951011165.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train.drop(columns=['2401','Borderlands'], inplace=True)


In [96]:
train.rename(
    columns={
        'Positive': 'Sentiment',
        'im getting on borderlands and i will murder you all ,': 'Tweets'
    },
    inplace=True
)

/tmp/ipykernel_88243/1106697295.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train.rename(


In [97]:
# print(train[train['Sentiment']=='Negative'])

In [98]:
# condition = (train['Sentiment']=='Positive') & (train['Sentiment'] == 'Negative') | (train['Sentiment'] =='Neutral')
# train = train[(train['Sentiment']=='Positive') & (train['Sentiment'] == 'Negative') | (train['Sentiment'] =='Neutral')]

In [99]:
train.head()

,Sentiment,Tweets
0,Positive,I am coming to the borders and I will kill you...
1,Positive,im getting on borderlands and i will kill you ...
2,Positive,im coming on borderlands and i will murder you...
3,Positive,im getting on borderlands 2 and i will murder ...
4,Positive,im getting into borderlands and i can murder y...


In [100]:
train['Sentiment'].value_counts()


Sentiment
Negative      21698
Positive      19712
Neutral       17708
Irrelevant    12537
Name: count, dtype: int64

In [101]:
train = train[train['Sentiment'] !='Irrelevant']


In [102]:
train['Sentiment'].value_counts()


Sentiment
Negative    21698
Positive    19712
Neutral     17708
Name: count, dtype: int64

feature Engineering : -> No. of characters, No. or words.

### Feature Encoding

In [103]:
from sklearn.preprocessing import LabelEncoder

In [104]:
le = LabelEncoder()

In [105]:
train['Sentiment'] = le.fit_transform(train['Sentiment'])

In [106]:
train['Sentiment']

0        2
1        2
2        2
3        2
4        2
        ..
74676    2
74677    2
74678    2
74679    2
74680    2
Name: Sentiment, Length: 59118, dtype: int64

In [107]:
train['Sentiment'].value_counts()

Sentiment
0    21698
2    19712
1    17708
Name: count, dtype: int64

In [108]:
from tensorflow import keras 

In [109]:
Tokenizer = keras.preprocessing.text.Tokenizer

tokenizer = Tokenizer()

### Here Keras Tokenizer is used to convert text into sequence of numbers. 
when we use tokenizer, it creates a dictionary of all the words in the text and assigns a unique number to each word. Then it converts the text into a sequence of numbers based on the dictionary.

- and also if new word comes in the test data which is not present in the training data, then it uses something called out of vocabulary token (OOV) to represent that word.
- Let's say : Hello, okay. is the training data. 
- Testing : amit --> OOV token --> treating as "Nothing" or "Unknown"

In [110]:
tokenizer = Tokenizer(oov_token="Nothing")

In [111]:
tokenizer.fit_on_texts(train['Tweets'])


In [112]:
tokenizer.word_index

{'Nothing': 1,
 'the': 2,
 'i': 3,
 'to': 4,
 'and': 5,
 'a': 6,
 'of': 7,
 'is': 8,
 'in': 9,
 'for': 10,
 'it': 11,
 'this': 12,
 'my': 13,
 'on': 14,
 'you': 15,
 'that': 16,
 'com': 17,
 'game': 18,
 'with': 19,
 'so': 20,
 'be': 21,
 'me': 22,
 'have': 23,
 'just': 24,
 'but': 25,
 'not': 26,
 'are': 27,
 'all': 28,
 'was': 29,
 'at': 30,
 'like': 31,
 '2': 32,
 'out': 33,
 'from': 34,
 'your': 35,
 'now': 36,
 'get': 37,
 'pic': 38,
 'twitter': 39,
 'as': 40,
 't': 41,
 'we': 42,
 'play': 43,
 'johnson': 44,
 'they': 45,
 'one': 46,
 'can': 47,
 'do': 48,
 'if': 49,
 'new': 50,
 'good': 51,
 'no': 52,
 'about': 53,
 'an': 54,
 'has': 55,
 'will': 56,
 'really': 57,
 'when': 58,
 'more': 59,
 'up': 60,
 "i'm": 61,
 'what': 62,
 'time': 63,
 'love': 64,
 'unk': 65,
 '3': 66,
 'or': 67,
 'how': 68,
 'why': 69,
 'by': 70,
 'some': 71,
 'co': 72,
 'shit': 73,
 "it's": 74,
 'been': 75,
 'amazon': 76,
 'still': 77,
 'people': 78,
 '’': 79,
 'https': 80,
 'facebook': 81,
 'games': 82,
 '

In [113]:
tokenizer.word_counts

OrderedDict([('i', 24421),
             ('am', 1517),
             ('coming', 403),
             ('to', 23771),
             ('the', 36098),
             ('borders', 11),
             ('and', 21712),
             ('will', 2684),
             ('kill', 348),
             ('you', 9359),
             ('all', 4492),
             ('im', 547),
             ('getting', 1046),
             ('on', 9756),
             ('borderlands', 1412),
             ('murder', 67),
             ('2', 3716),
             ('me', 5527),
             ('into', 947),
             ('can', 2904),
             ('so', 6410),
             ('spent', 179),
             ('a', 19236),
             ('few', 467),
             ('hours', 481),
             ('making', 527),
             ('something', 752),
             ('for', 12494),
             ('fun', 1329),
             ('if', 2857),
             ("don't", 1337),
             ('know', 1391),
             ('huge', 310),
             ('fan', 228),
             ('maya', 22),
 

In [114]:
tokenizer.document_count

59118

In [115]:
len((tokenizer.word_index)) ## Vocabulary size

29229

In [116]:
tokenizer.word_counts

OrderedDict([('i', 24421),
             ('am', 1517),
             ('coming', 403),
             ('to', 23771),
             ('the', 36098),
             ('borders', 11),
             ('and', 21712),
             ('will', 2684),
             ('kill', 348),
             ('you', 9359),
             ('all', 4492),
             ('im', 547),
             ('getting', 1046),
             ('on', 9756),
             ('borderlands', 1412),
             ('murder', 67),
             ('2', 3716),
             ('me', 5527),
             ('into', 947),
             ('can', 2904),
             ('so', 6410),
             ('spent', 179),
             ('a', 19236),
             ('few', 467),
             ('hours', 481),
             ('making', 527),
             ('something', 752),
             ('for', 12494),
             ('fun', 1329),
             ('if', 2857),
             ("don't", 1337),
             ('know', 1391),
             ('huge', 310),
             ('fan', 228),
             ('maya', 22),
 

In [117]:
sequences = tokenizer.texts_to_sequences(train['Tweets'])


In [118]:
sequences

[[3, 114, 400, 4, 2, 6290, 5, 3, 56, 455, 15, 28],
 [314, 171, 14, 121, 5, 3, 56, 455, 15, 28],
 [314, 400, 14, 121, 5, 3, 56, 1737, 15, 28],
 [314, 171, 14, 121, 32, 5, 3, 56, 1737, 15, 22, 28],
 [314, 171, 191, 121, 5, 3, 47, 1737, 15, 28],
 [20,
  3,
  798,
  6,
  361,
  347,
  324,
  240,
  10,
  133,
  49,
  15,
  130,
  125,
  3,
  114,
  6,
  496,
  121,
  629,
  5,
  3924,
  8,
  46,
  7,
  13,
  328,
  749,
  20,
  3,
  739,
  4,
  123,
  491,
  6,
  5328,
  10,
  13,
  282,
  145,
  8,
  2,
  762,
  2053,
  5577,
  2,
  8859,
  3,
  213,
  425,
  38,
  39,
  17,
  15636],
 [20,
  3,
  798,
  6,
  1012,
  7,
  347,
  308,
  240,
  10,
  133,
  49,
  15,
  130,
  125,
  16,
  61,
  6,
  496,
  121,
  629,
  5,
  3924,
  8,
  46,
  7,
  13,
  328,
  749,
  3,
  739,
  4,
  123,
  6,
  5328,
  10,
  13,
  282,
  1189,
  2,
  762,
  1467,
  1819,
  4,
  2,
  8859,
  3,
  213,
  23,
  133,
  38,
  39,
  17,
  15636],
 [20,
  3,
  798,
  6,
  361,
  347,
  308,
  240,
  10,
  133,
 

In [119]:
len((sequences[0]))

12

In [120]:
len(sequences[1])

10

In [121]:
len(sequences[2])

10

In [122]:
from keras.utils import pad_sequences

### Padding the sequences :
- When we convert text into sequence of numbers, we get sequences of different lengths.
- But for training a model, we need all the sequences to be of the same length.
- So we use padding to make all the sequences of the same length.
- pad as pre or post. 
- If we pad as pre, then we add zeros at the beginning of the sequence.
- If we pad as post, then we add zeros at the end of the sequence.

In [123]:
sequences = pad_sequences(sequences, padding='pre')

In [124]:
sequences

array([[   0,    0,    0, ...,  455,   15,   28],
       [   0,    0,    0, ...,  455,   15,   28],
       [   0,    0,    0, ..., 1737,   15,   28],
       ...,
       [   0,    0,    0, ...,  154, 1028, 2541],
       [   0,    0,    0, ...,   79,   41, 2541],
       [   0,    0,    0, ...,    3, 1028, 2541]], dtype=int32)

### Text --> [Integer Sequence] --> [Padded Integer Sequence] --> Model

In [125]:
from keras.layers import SimpleRNN, Embedding, Dense,Input
from keras.models import Sequential

In [126]:
X = sequences

In [127]:
y = train['Sentiment'].values

In [128]:
y

array([2, 2, 2, ..., 2, 2, 2])

In [129]:
from sklearn.model_selection import train_test_split

In [130]:
x_train, x_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [131]:
x_train.shape

(47294, 166)

### Embedding Layer :
- It is used to convert the integer sequences into dense vectors of fixed size.
- After including the embedding layer, we capture the semantic meaning of the words in the text.

In [132]:
model = Sequential()
model.add(SimpleRNN(32,input_shape=(166,1),return_sequences=False))
model.add(Dense(3, activation='softmax')) 
## softmax is used for multi-class classification


/home/dharmender/anaconda3/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [133]:
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_3 (SimpleRNN)        │ (None, 32)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,187 (4.64 KB)

 Trainable params: 1,187 (4.64 KB)

 Non-trainable params: 0 (0.00 B)

In [134]:
## For multi-class classification, we use 'sparse_categorical_crossentropy' as the loss function and 'adam' as the optimizer. 
# We also track the accuracy metric during training.
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


In [135]:
model.fit(x_train, y_train, epochs=10, validation_data=(x_test, y_test), batch_size=64)

Epoch 1/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.3571 - loss: 1.1141 - val_accuracy: 0.3737 - val_loss: 1.0920
Epoch 2/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3625 - loss: 1.0951 - val_accuracy: 0.3721 - val_loss: 1.0934
Epoch 3/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3630 - loss: 1.0953 - val_accuracy: 0.3736 - val_loss: 1.0931
Epoch 4/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3622 - loss: 1.0960 - val_accuracy: 0.3606 - val_loss: 1.0973
Epoch 5/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3556 - loss: 1.0979 - val_accuracy: 0.3731 - val_loss: 1.0976
Epoch 6/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3650 - loss: 1.0960 - val_accuracy: 0.3384 - val_loss: 1.0970
Epoch 7/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3590 - loss: 1.0965 - val_accuracy: 0.3408 - val_loss: 1.0960
Epoch 8/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3644 - loss: 1.0955 - val_accuracy: 0.

### Vocab Size :
- It is the total number of unique words in the text.
- It is calculated as the length of the word index created by the tokenizer + 1 (to account for the OOV token).

In [136]:
vocab_size = len(tokenizer.word_index) + 1


In [137]:
vocab_size

29230

In [138]:
model_2 = Sequential()
model_2.add(Input(shape=(166,)))
model_2.add(Embedding(input_dim=vocab_size, output_dim=64)) ## input_length is not required as the input shape is already defined in the first layer
model_2.add(Dense(3, activation='softmax')) 

In [139]:
model_2.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 166, 64)        │     1,870,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 166, 3)         │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,870,915 (7.14 MB)

 Trainable params: 1,870,915 (7.14 MB)

 Non-trainable params: 0 (0.00 B)

In [140]:
model_final = Sequential([
    Input(shape=(166,)),
    Embedding(input_dim=vocab_size, output_dim=64, mask_zero=True),
    SimpleRNN(32),
    Dense(3, activation='softmax')
])


In [141]:
model_final.compile(optimizer='adam', 
                    loss='sparse_categorical_crossentropy', 
                    metrics=['accuracy']
                    )

In [142]:
model_final.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 166, 64)        │     1,870,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,873,923 (7.15 MB)

 Trainable params: 1,873,923 (7.15 MB)

 Non-trainable params: 0 (0.00 B)

In [143]:
model_final.fit(x_train, y_train, epochs=100, validation_data=(x_test, y_test), batch_size=64)

Epoch 1/100
739/739 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.6307 - loss: 0.8092 - val_accuracy: 0.8617 - val_loss: 0.3583
Epoch 2/100
739/739 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.9295 - loss: 0.1927 - val_accuracy: 0.8950 - val_loss: 0.2887
Epoch 3/100
739/739 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.9614 - loss: 0.0960 - val_accuracy: 0.8976 - val_loss: 0.2943
Epoch 4/100
739/739 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.9724 - loss: 0.0652 - val_accuracy: 0.8519 - val_loss: 0.4318
Epoch 5/100
739/739 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.9673 - loss: 0.0803 - val_accuracy: 0.8743 - val_loss: 0.3833
Epoch 6/100
739/739 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.9738 - loss: 0.0583 - val_accuracy: 0.8897 - val_loss: 0.3747
Epoch 7/100
739/739 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.9781 - loss: 0.0482 - val_accuracy: 0.8879 - val_loss: 0.3860
Epoch 8/100
739/739 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.9788 - loss: 0.0436 - val_acc

In [144]:
import pickle 

In [145]:
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

In [146]:
model_final.save('sentiment_analysis_model.h5')
model_final.save('sentiment_analysis_model.keras')